In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Tuple, Optional

# ============================================================
# Verbose helpers
# ============================================================
class VerboseTracer:
    def __init__(self, enabled: bool = True):
        self.enabled = enabled
        self.step = 0

    def log(self, msg: str, t: Optional[torch.Tensor] = None, level: Optional[int] = None):
        if not self.enabled:
            return
        self.step += 1
        prefix = f"[{self.step:04d}]"
        lvl = f" | level={level}" if level is not None else ""
        if t is None:
            print(f"{prefix}{lvl} {msg}")
        else:
            # show (H,W,C) and also full (B,C,H,W)
            b, c, h, w = t.shape
            print(f"{prefix}{lvl} {msg} | HxW={h}x{w} | C={c} | tensor={tuple(t.shape)} | device={t.device} | dtype={t.dtype}")

def fmt_shape(t: torch.Tensor) -> str:
    b, c, h, w = t.shape
    return f"(B={b}, C={c}, H={h}, W={w})"


# ============================================================
# Basic building blocks (verbose)
# ============================================================
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, p: int = 1, name: str = ""):
        super().__init__()
        self.name = name or f"ConvBNReLU({in_ch}->{out_ch})"
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=k, padding=p, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.act  = nn.ReLU(inplace=True)

    def forward(self, x, tr: Optional[VerboseTracer] = None, level: Optional[int] = None):
        if tr: tr.log(f"{self.name}: INPUT", x, level)
        y = self.conv(x)
        if tr: tr.log(f"{self.name}: after Conv2d(k={self.conv.kernel_size}, pad={self.conv.padding})", y, level)
        y = self.bn(y)
        if tr: tr.log(f"{self.name}: after BatchNorm2d", y, level)
        y = self.act(y)
        if tr: tr.log(f"{self.name}: after ReLU", y, level)
        return y


class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, name: str = ""):
        super().__init__()
        base = name or f"DoubleConv({in_ch}->{out_ch})"
        self.name = base
        self.c1 = ConvBNReLU(in_ch, out_ch, name=base + "/conv1")
        self.c2 = ConvBNReLU(out_ch, out_ch, name=base + "/conv2")

    def forward(self, x, tr: Optional[VerboseTracer] = None, level: Optional[int] = None):
        if tr: tr.log(f"{self.name}: INPUT", x, level)
        y = self.c1(x, tr=tr, level=level)
        y = self.c2(y, tr=tr, level=level)
        if tr: tr.log(f"{self.name}: OUTPUT", y, level)
        return y


def center_crop_to(x: torch.Tensor, ref: torch.Tensor, tr: Optional[VerboseTracer] = None, level: Optional[int] = None):
    if x.shape[-2:] == ref.shape[-2:]:
        if tr: tr.log("center_crop_to: no crop needed", x, level)
        return x
    dh = x.shape[-2] - ref.shape[-2]
    dw = x.shape[-1] - ref.shape[-1]
    if tr:
        tr.log(f"center_crop_to: cropping (dh={dh}, dw={dw}) to match ref HxW={ref.shape[-2:]}",
               x, level)
    return x[:, :, dh // 2 : dh // 2 + ref.shape[-2], dw // 2 : dw // 2 + ref.shape[-1]]


class UpBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, name: str = ""):
        super().__init__()
        self.name = name or f"UpBlock({in_ch}->{out_ch}, skip={skip_ch})"
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2, bias=False)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch, name=self.name + "/postcat")

    def forward(self, x, skip, tr: Optional[VerboseTracer] = None, level_in: Optional[int] = None, level_out: Optional[int] = None):
        # x is at level_in, output becomes level_out (= level_in - 1 usually)
        if tr: tr.log(f"{self.name}: INPUT x", x, level_in)
        if tr: tr.log(f"{self.name}: INPUT skip", skip, level_out)

        y = self.up(x)
        if tr: tr.log(f"{self.name}: after ConvTranspose2d(stride=2)", y, level_out)

        skip2 = center_crop_to(skip, y, tr=tr, level=level_out)

        cat = torch.cat([skip2, y], dim=1)
        if tr: tr.log(f"{self.name}: after cat([skip, up]) dim=1", cat, level_out)

        out = self.conv(cat, tr=tr, level=level_out)
        if tr: tr.log(f"{self.name}: OUTPUT", out, level_out)
        return out


# ============================================================
# Asymmetric mini-U-Net block (verbose)
# ============================================================
class AsymUNetBlock(nn.Module):
    """
    Descend from start_level to level 4, then ascend to target_level.
    Uses:
      - internal skips for levels >= start_level (created during this block's descent)
      - global skips for levels < start_level (created elsewhere)
    """
    def __init__(self, channels, start_level: int, target_level: int, name: str = ""):
        super().__init__()
        assert 0 <= start_level <= 3
        assert 0 <= target_level <= 3
        self.channels = list(channels)
        self.start = start_level
        self.target = target_level
        self.name = name or f"AsymUNetBlock(start={start_level}, target={target_level})"

        self.pool = nn.MaxPool2d(2)

        # Down: map C_l -> C_{l+1} for l=start..3
        self.down_convs = nn.ModuleList([
            DoubleConv(self.channels[l], self.channels[l + 1], name=f"{self.name}/down_l{l}_to_l{l+1}")
            for l in range(start_level, 4)
        ])

        self.bottom = DoubleConv(self.channels[4], self.channels[4], name=f"{self.name}/bottom_l4")

        # Up: 4 -> target (needs skips at levels 3..target)
        self.up_blocks = nn.ModuleList([
            UpBlock(in_ch=self.channels[l], skip_ch=self.channels[l - 1], out_ch=self.channels[l - 1],
                    name=f"{self.name}/up_l{l}_to_l{l-1}")
            for l in range(4, target_level, -1)
        ])

    def forward(self, x, global_skips: Dict[int, torch.Tensor], tr: Optional[VerboseTracer] = None):
        if tr:
            tr.log(f"{self.name}: START (enter at level {self.start})", x, self.start)

        internal_skips = {self.start: x}
        if tr:
            tr.log(f"{self.name}: internal_skips[{self.start}] = x", x, self.start)

        cur = x

        # DESCENT: start -> 4
        for idx, l in enumerate(range(self.start, 4)):
            # pool: level l -> l+1
            cur = self.pool(cur)
            if tr: tr.log(f"{self.name}: MaxPool2d(2) level {l}->{l+1}", cur, l + 1)

            # conv at level l+1
            cur = self.down_convs[idx](cur, tr=tr, level=l + 1)
            internal_skips[l + 1] = cur
            if tr: tr.log(f"{self.name}: internal_skips[{l+1}] stored", cur, l + 1)

        # bottom at level 4
        cur = self.bottom(cur, tr=tr, level=4)
        if tr: tr.log(f"{self.name}: after bottom (still level 4)", cur, 4)

        # ASCENT: 4 -> target
        level = 4
        for up in self.up_blocks:
            next_level = level - 1

            # choose skip: internal first, else global
            skip = internal_skips.get(next_level, None)
            src = "internal"
            if skip is None:
                skip = global_skips.get(next_level, None)
                src = "global"

            if skip is None:
                raise KeyError(
                    f"{self.name}: Missing skip for level {next_level}. "
                    f"Internal: {sorted(internal_skips.keys())}, Global: {sorted(global_skips.keys())}"
                )

            if tr:
                tr.log(f"{self.name}: using {src}_skip at level {next_level}", skip, next_level)

            cur = up(cur, skip, tr=tr, level_in=level, level_out=next_level)
            level = next_level

        if tr:
            tr.log(f"{self.name}: END (exit at level {self.target})", cur, self.target)
        return cur


class DownToBottleneck(nn.Module):
    def __init__(self, channels, name: str = "DownToBottleneck"):
        super().__init__()
        self.name = name
        self.pool = nn.MaxPool2d(2)
        self.to4  = DoubleConv(channels[3], channels[4], name=f"{name}/to_l4")
        self.bot  = DoubleConv(channels[4], channels[4], name=f"{name}/bot_l4")

    def forward(self, x3, tr: Optional[VerboseTracer] = None):
        if tr: tr.log(f"{self.name}: INPUT (level 3)", x3, 3)
        x4 = self.pool(x3)
        if tr: tr.log(f"{self.name}: MaxPool2d(2) level 3->4", x4, 4)
        x4 = self.to4(x4, tr=tr, level=4)
        x4 = self.bot(x4, tr=tr, level=4)
        if tr: tr.log(f"{self.name}: OUTPUT (level 4)", x4, 4)
        return x4


# ============================================================
# UU-Net (verbose forward)
# ============================================================
class UUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2, channels=(64, 128, 256, 512, 1024)):
        super().__init__()
        self.channels = list(channels)

        self.stem = DoubleConv(in_channels, self.channels[0], name="STEM_l0")

        # Phase A
        self.a0 = AsymUNetBlock(self.channels, start_level=0, target_level=1, name="A0_0to4to1")
        self.a1 = AsymUNetBlock(self.channels, start_level=1, target_level=2, name="A1_1to4to2")
        self.a2 = AsymUNetBlock(self.channels, start_level=2, target_level=3, name="A2_2to4to3")
        self.a3 = DownToBottleneck(self.channels, name="A3_3to4")

        # Phase B
        self.up4_to_3 = UpBlock(in_ch=self.channels[4], skip_ch=self.channels[3], out_ch=self.channels[3], name="B0_up4to3")
        self.b3 = AsymUNetBlock(self.channels, start_level=3, target_level=2, name="B3_3to4to2")
        self.b2 = AsymUNetBlock(self.channels, start_level=2, target_level=1, name="B2_2to4to1")
        self.b1 = AsymUNetBlock(self.channels, start_level=1, target_level=0, name="B1_1to4to0")

        self.out_conv = nn.Conv2d(self.channels[0], out_channels, kernel_size=1)

    def forward(self, x, verbose: bool = True):
        tr = VerboseTracer(enabled=verbose)

        tr.log("MODEL INPUT", x, level=0)

        global_skips: Dict[int, torch.Tensor] = {}

        # level 0
        x0 = self.stem(x, tr=tr, level=0)
        global_skips[0] = x0
        tr.log("global_skips[0] stored", x0, level=0)

        # Phase A
        x1 = self.a0(x0, global_skips, tr=tr)   # exits at level 1
        global_skips[1] = x1
        tr.log("global_skips[1] stored", x1, level=1)

        x2 = self.a1(x1, global_skips, tr=tr)   # exits at level 2
        global_skips[2] = x2
        tr.log("global_skips[2] stored", x2, level=2)

        x3 = self.a2(x2, global_skips, tr=tr)   # exits at level 3
        global_skips[3] = x3
        tr.log("global_skips[3] stored", x3, level=3)

        x4 = self.a3(x3, tr=tr)                 # level 4

        # Phase B
        tr.log("PHASE B: step B0 (4->3)", x4, level=4)
        y3 = self.up4_to_3(x4, global_skips[3], tr=tr, level_in=4, level_out=3)

        tr.log("PHASE B: step B3 (3->4->2)", y3, level=3)
        y2 = self.b3(y3, global_skips, tr=tr)   # exits at level 2

        tr.log("PHASE B: step B2 (2->4->1)", y2, level=2)
        y1 = self.b2(y2, global_skips, tr=tr)   # exits at level 1

        tr.log("PHASE B: step B1 (1->4->0)", y1, level=1)
        y0 = self.b1(y1, global_skips, tr=tr)   # exits at level 0

        tr.log("HEAD: out_conv 1x1 (to logits)", y0, level=0)
        out = self.out_conv(y0)
        tr.log("MODEL OUTPUT (logits)", out, level=0)

        return out


# ============================================================
# Example: apply to ONE image tensor
# ============================================================
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = UUNet(in_channels=3, out_channels=2).to(device)
    model.eval()

    # Example input: one RGB image 608x608
    x = torch.randn(1, 3, 608, 608, device=device).to(device)

    with torch.no_grad():
        y = model(x, verbose=True)

    print("Final logits shape:", y.shape)  # (1, 2, 608, 608)
    # If you want probabilities:
    # probs = torch.softmax(y, dim=1)  # (1,2,H,W)
    # vessel_prob = probs[:, 1]        # (1,H,W)


/home/usrs/hnoel/miniconda3/envs/iic/lib/python3.10/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_37 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
